#### Gold: project-grain classification table

Builds ml_project_classification: ONE row per completed project, with only
features KNOWN AT PROJECT START (leakage-safe), and a binary label for whether
the project ended up over-cost. Feeds the over-cost risk CLASSIFIER (notebook 06).

Run this AFTER 04_gold_star_schema (it reads the gold dim/fact tables).

WHY THIS TABLE EXISTS (design rationale):
  - GRAIN: project-level, not line-item. The business question is "is THIS
    project at risk of overrunning?" -> one prediction per project.
  - LEAKAGE DISCIPLINE: features use only what's known at kickoff (contract,
    scope, delivery, planned schedule, budget mix). NO actuals, NO realized
    durations/costs/incidents -- those don't exist for a live project.
  - LABEL: is_overrun = 1 if project total actual/budget > THRESHOLD.
    THRESHOLD = 1.18, set near the project-level median so the split reflects
    "worse than a typical project" and preserves the natural base rate (no
    resampling -> probabilities stay calibrated).


#### Cell 1

In [1]:
# Config + load gold
from pyspark.sql import functions as F
from pyspark.sql import types as T

# Over-cost threshold at PROJECT grain. Set near the project-level median of
# overrun_ratio so the label means "above-median overrun" and the class balance
# is natural (not forced). Raising this flags only more extreme overruns.
OVERRUN_THRESHOLD = 1.18

dim_project = spark.table("dim_project")
fact_cost   = spark.table("fact_cost")
print(f"threshold = {OVERRUN_THRESHOLD}")


StatementMeta(, f07d4977-e2cb-420b-ab32-0be5ca4f1a93, 3, Finished, Available, Finished, False)

threshold = 1.18


#### Cell 2

In [2]:
# Aggregate cost to PROJECT grain (compute the outcome + budget mix)
# Only valid cost lines (positive budget & actual) contribute.
valid_cost = fact_cost.filter(F.col("actual_valid") &
                              (F.col("budget_amount") > 0) &
                              (F.col("actual_amount") > 0))

# project totals -> the outcome (project_overrun) and total_budget (a start-known feature)
proj_totals = (valid_cost.groupBy("project_sk")
    .agg(F.sum("budget_amount").alias("total_budget"),
         F.sum("actual_amount").alias("total_actual"),
         F.count("*").alias("n_cost_lines"))
    .withColumn("project_overrun", F.col("total_actual") / F.col("total_budget")))

# MEP budget share -- known from the estimate at kickoff (leakage-safe).
# division_sk -> csi_division via dim_division.
dim_division = spark.table("dim_division")
cost_div = valid_cost.join(dim_division.select("division_sk", "csi_division"), "division_sk", "left")
mep_share = (cost_div
    .withColumn("is_mep", F.col("csi_division").isin("21", "22", "23", "26", "27"))
    .groupBy("project_sk")
    .agg((F.sum(F.when(F.col("is_mep"), F.col("budget_amount")).otherwise(0.0)) /
          F.sum("budget_amount")).alias("pct_budget_mep")))

proj_agg = proj_totals.join(mep_share, "project_sk", "left")
print(f"projects with computable outcome: {proj_agg.count()}")


StatementMeta(, f07d4977-e2cb-420b-ab32-0be5ca4f1a93, 4, Finished, Available, Finished, False)

projects with computable outcome: 240


#### Cell 3

In [3]:
# Assemble features (start-known only) + binary label
# Pull start-known attributes from dim_project. Exclude anything realized during
# execution. planned_duration comes from planned dates; is_winter_start is known.
proj_attrs = (dim_project
    .withColumn("planned_duration_days",
                F.datediff(F.col("planned_end_date"), F.col("start_date")))
    .select("project_sk", "project_id", "project_type", "delivery_method", "region",
            "contract_value", "square_footage", "planned_duration_days",
            "is_winter_start", "status", "data_split"))

ml_project = (proj_agg.join(proj_attrs, "project_sk", "left")
    # binary label from the project-level outcome
    .withColumn("is_overrun",
                F.when(F.col("project_overrun") > OVERRUN_THRESHOLD, 1).otherwise(0))
    # log-scale contract value (spans orders of magnitude) -- a modeling nicety
    .withColumn("log_contract_value",
                F.when(F.col("contract_value") > 0, F.log(F.col("contract_value"))))
    .select(
        "project_id", "project_sk",
        # ---- FEATURES (all known at project start) ----
        "project_type", "delivery_method", "region",
        "contract_value", "log_contract_value", "square_footage",
        "planned_duration_days", "is_winter_start", "pct_budget_mep",
        # ---- LABEL + context ----
        "is_overrun", "project_overrun",     # keep raw ratio for reporting/validation
        "total_budget", "total_actual", "n_cost_lines",
        "status", "data_split"))
        

StatementMeta(, f07d4977-e2cb-420b-ab32-0be5ca4f1a93, 5, Finished, Available, Finished, False)

#### Cell 4

In [4]:
# Write the classification table
(ml_project.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable("ml_project_classification"))
print(f"ml_project_classification written: {ml_project.count()} projects")

StatementMeta(, f07d4977-e2cb-420b-ab32-0be5ca4f1a93, 6, Finished, Available, Finished, False)

ml_project_classification written: 240 projects


#### Cell 5

In [5]:
# Validate: class balance + signal survival at project grain
print(f"=== CLASS BALANCE at threshold {OVERRUN_THRESHOLD} ===")
bal = ml_project.groupBy("is_overrun").count().collect()
tot = ml_project.count()
for r in sorted(bal, key=lambda x: x["is_overrun"]):
    label = "over-cost (1)" if r["is_overrun"] == 1 else "on-budget (0)"
    print(f"  {label}: {r['count']} ({r['count']/tot*100:.1f}%)")

print(f"\n=== SIGNAL AT PROJECT GRAIN ===")
print("Over-cost RATE by delivery method (engineered: DBB high, IPD low):")
(ml_project.groupBy("delivery_method")
    .agg(F.round(F.mean("is_overrun"), 3).alias("overrun_rate"),
         F.count("*").alias("n"))
    .orderBy(F.col("overrun_rate").desc())
    .show(truncate=False))

print("Mean pct_budget_mep by outcome (engineered: MEP-heavy -> more overrun):")
(ml_project.groupBy("is_overrun")
    .agg(F.round(F.mean("pct_budget_mep"), 3).alias("avg_mep_share"))
    .show())

print("Provenance split available:")
ml_project.groupBy("data_split").count().show()

StatementMeta(, f07d4977-e2cb-420b-ab32-0be5ca4f1a93, 7, Finished, Available, Finished, False)

=== CLASS BALANCE at threshold 1.18 ===
  on-budget (0): 109 (45.4%)
  over-cost (1): 131 (54.6%)

=== SIGNAL AT PROJECT GRAIN ===
Over-cost RATE by delivery method (engineered: DBB high, IPD low):
+----------------+------------+---+
|delivery_method |overrun_rate|n  |
+----------------+------------+---+
|Design-Bid-Build|0.971       |35 |
|CM Agency       |0.868       |53 |
|CM at Risk      |0.511       |45 |
|Design-Build    |0.348       |46 |
|IPD             |0.197       |61 |
+----------------+------------+---+

Mean pct_budget_mep by outcome (engineered: MEP-heavy -> more overrun):
+----------+-------------+
|is_overrun|avg_mep_share|
+----------+-------------+
|         1|        0.331|
|         0|        0.286|
+----------+-------------+

Provenance split available:
+----------+-----+
|data_split|count|
+----------+-----+
|     train|  120|
|      test|  120|
+----------+-----+



In [6]:
from pyspark.sql import functions as F
spark.table("ml_project_classification").filter(F.col("data_split")=="test").groupBy("delivery_method").agg(
    F.round(F.mean("project_overrun"),3).alias("mean_overrun"),
    F.count("*").alias("n")
).orderBy(F.col("mean_overrun").desc()).show(truncate=False)

StatementMeta(, f07d4977-e2cb-420b-ab32-0be5ca4f1a93, 8, Finished, Available, Finished, False)

+----------------+------------+---+
|delivery_method |mean_overrun|n  |
+----------------+------------+---+
|Design-Bid-Build|1.316       |22 |
|CM Agency       |1.244       |26 |
|CM at Risk      |1.178       |21 |
|Design-Build    |1.146       |25 |
|IPD             |1.116       |26 |
+----------------+------------+---+

